## 11.1 GRU - 核心结构

#### 1、什么是 GRU

##### 1.1 GRU 的全称
GRU 的全称是 Gated Recurrent Unit，中文通常翻译为 门控循环单元。

它也是一种 **专门为序列数据设计的循环神经网络结构**，本质上属于 RNN 家族的一员。

所以从分类上看：

- 普通 RNN：最基础的循环结构
- LSTM：为了解决长期依赖问题而设计的更复杂结构
- GRU：在 LSTM 基础上进一步简化出来的门控结构

也就是说，GRU 可以理解为：

一种比普通 RNN 更强、比 LSTM 更简洁的门控循环网络。


##### 1.2 GRU 的必要性
我们刚刚学完 LSTM，已经知道：

- 普通 RNN 容易出现长期依赖问题
- LSTM 通过门控机制和细胞状态，显著改善了这个问题

但是 LSTM 也有一个明显特点：

- 结构比较复杂
- 参数较多
- 计算量更大
- 理解和实现都更繁琐

于是研究者会继续思考一个问题：

能不能保留“门控机制”的优点，同时把结构做得更简单一些？

GRU 就是在这个思路下提出的。


##### 1.3 GRU 的目标是什么
GRU 的目标可以概括为一句话：

**用更简单的门控结构，达到接近 LSTM 的效果。**

所以 GRU 的设计理念不是完全推翻 LSTM，而是做“简化版优化”：

- 保留门控思想 ✅
- 保留对长期依赖的建模能力 ✅
- 去掉一些复杂部分 ✅
- 让参数更少、训练更快 ✅

#### 2、GRU 和 LSTM 的关系

##### 2.1 GRU 可以看成 LSTM 的简化版
这是学习 GRU 时最重要的起点。

如果你已经理解了 LSTM，那么 GRU 就不会难，因为它的核心思想并没有变：

- 都想解决普通 RNN 的长期依赖问题
- 都使用门控机制控制信息流
- 都通过“有选择地保留和更新信息”来增强记忆能力

区别在于：

- LSTM 更精细
- GRU 更简洁


##### 2.2 GRU 简化了什么
和 LSTM 相比，GRU 主要做了两个大简化：

**（1）没有单独的细胞状态 $C_t$**

LSTM 有两套状态：

- 隐藏状态 $h_t$
- 细胞状态 $C_t$

而 GRU 没有把这两者分开管理，而是类似于简单 RNN，它只有一个状态：

$h_t$

也就是说，在 GRU 中：

- “长期记忆”
- “当前输出状态”

被统一到一个状态里管理了。

**（2）门的数量更少**

LSTM 有三个核心门：

- 遗忘门 Forget Gate
- 输入门 Input Gate
- 输出门 Output Gate

而 GRU 通常只有两个核心门：

- 更新门 Update Gate
- 重置门 Reset Gate

所以从结构上看，GRU 比 LSTM 更轻量。

#### 3、GRU 的核心思想

##### 3.1 GRU 的目标
GRU 的本质目标和 LSTM 一样，仍然是在解决下面这个问题：

当前时刻，到底应该保留多少旧信息，又应该接收多少新信息？

所以 GRU 的核心不是“复杂公式”，而是一个非常朴素的信息管理思想：

- 旧信息不是全部保留
- 新信息不是全部接受
- 而是通过门控机制，有选择地融合


##### 3.2 GRU 的核心机制
GRU 可以概括成这样一个过程：

- 先决定要不要忘掉一部分旧信息
- 再决定当前输入要不要强烈参与
- 最后把旧状态和新候选状态按比例混合，得到新的隐藏状态

也就是说，GRU 其实是在做：

旧状态 $h_{t-1}$ 和新候选状态 $\tilde{h}_t$ 的加权融合。


#### 4、GRU 的核心结构总览

##### 4.1 GRU 在单时间步接收什么
和简单 RNN 一样。

在某一个时间步 $t$，GRU 接收两个主要输入：

- 当前输入：$x_t$
- 上一时刻隐藏状态：$h_{t-1}$

注意这里没有：

- 上一时刻细胞状态 $C_{t-1}$

因为 GRU 根本没有单独的细胞状态。

所以单时间步可以写成：

$(x_t,\ h_{t-1}) \rightarrow h_t$


##### 4.2 GRU 在单时间步输出什么
与简单 RNN 一样。

GRU 只输出一个状态：

$h_t$

这个状态既：

- 是当前时间步的输出
- 也是下一时间步继续传递的状态

所以 GRU 的信息流比 LSTM 更紧凑。


##### 4.3 为什么说 GRU 的状态更“统一”
在 LSTM 里：

- $C_t$ 更像长期记忆
- $h_t$ 更像当前对外输出

而 GRU 不再区分这两种角色，而是把它们合并在一起：

$h_t$

所以你可以理解为：

GRU 用一个统一的隐藏状态，同时承担“记忆”和“输出”两种职责。

这也是它比 LSTM 更简洁的根本原因。

#### 5、GRU 的两个核心门

##### 5.1 更新门 Update Gate

**作用**
更新门通常记作：

$z_t$

它决定的是：

当前时刻，是更应该保留旧状态，还是更应该接收新状态。

公式通常写作：

$z_t = \sigma(W_z[h_{t-1}, x_t] + b_z)$

其中：

- 激活函数是 Sigmoid
- 输出范围在 $(0,1)$
- $z_t$ 的每个元素都表示一个“更新比例”

它的直觉理解非常重要：

- $z_t$ 接近 $1$：更倾向于保留旧状态
- $z_t$ 接近 $0$：更倾向于使用新候选状态

所以更新门本质上是在回答：

**这一维信息，旧的还重要吗？**


##### 5.2 重置门 Reset Gate

**作用**
重置门通常记作：

$r_t$

它决定的是：

在生成候选新状态时，上一时刻隐藏状态要参与多少。

公式通常写作：

$r_t = \sigma(W_r[h_{t-1}, x_t] + b_r)$

它的作用重点不在最终输出，而在于：

影响“候选状态”的生成过程。

直观理解：

- $r_t$ 接近 $0$：过去状态影响减弱，像“先忘掉一部分过去再生成新内容”
- $r_t$ 接近 $1$：过去状态完整参与候选状态生成

所以重置门更像一个：

**过去信息参与程度控制器**

#### 6、候选隐藏状态是什么

##### 6.1 候选隐藏状态的作用
除了两个门之外，GRU 还会生成一个 候选隐藏状态，通常记作：

$\tilde{h}_t$

它表示：

如果当前时刻要写入新的信息，那么新信息长什么样。

公式通常写作：

$\tilde{h}_t = \tanh(W_h[r_t \odot h_{t-1}, x_t] + b_h)$

这里最关键的是：

$r_t \odot h_{t-1}$

说明上一时刻隐藏状态不是直接参与，而是先经过 **重置门筛选** 后，再参与候选状态生成。


##### 6.2 候选隐藏状态的直观理解
候选隐藏状态不是最终输出，它只是：

**当前时刻准备写进去的新内容**

所以你可以把它理解成：

- 更新门 $z_t$：决定要不要更新
- 重置门 $r_t$：决定过去信息怎么参与生成新内容
- 候选状态 $\tilde{h}_t$：真正生成出来的新内容

#### 7、GRU 如何得到最终隐藏状态

##### 7.1 最核心的更新公式
GRU 最关键的公式是：

$h_t = z_t \odot h_{t-1} + (1 - z_t)\odot \tilde{h}_t$

这个公式一定要反复理解。⭐

它表示新的隐藏状态由两部分组成：

- 保留下来的旧状态
- 新生成的候选状态


##### 7.2 这个公式表达了什么思想
这个公式本质上表达的是：

当前隐藏状态不是“全盘推翻重算”，而是“旧状态和新状态的加权融合”。

这点和 LSTM 非常像，但实现方式更简洁：

- LSTM：通过细胞状态 + 多个门做控制
- GRU：直接在旧隐藏状态和新候选状态之间做混合


##### 7.3 更新门的意义再看一遍
在这个公式里，更新门 $z_t$ 的含义最容易真正看懂：

- $z_t \rightarrow 1$：$h_t \approx h_{t-1}$，更多保留过去
- $z_t \rightarrow 0$：$h_t \approx \tilde{h}_t$，更多采用新内容

所以更新门本质上就是一个：

**旧记忆 vs 新记忆 的平衡器**

#### 8、GRU 的完整单时间步流程

##### 8.1 顺序梳理
在时间步 $t$，GRU 的前向传播通常按下面顺序进行：

**第一步：计算更新门**

$z_t = \sigma(W_z[h_{t-1}, x_t] + b_z)$

决定最终状态更偏向旧状态还是新状态。

**第二步：计算重置门**

$r_t = \sigma(W_r[h_{t-1}, x_t] + b_r)$

决定过去状态在生成候选状态时参与多少。

**第三步：生成候选隐藏状态**

$\tilde{h}_t = \tanh(W_h[r_t \odot h_{t-1}, x_t] + b_h)$

生成当前时刻的新内容候选。

**第四步：融合得到当前隐藏状态**

$h_t = z_t \odot h_{t-1} + (1-z_t)\odot \tilde{h}_t$

得到当前时刻最终隐藏状态。


##### 8.2 一句话概括整个流程
先决定过去要保留多少，再决定过去如何参与生成新内容，最后把旧状态和新内容融合起来。

#### 9、GRU 和 LSTM 的结构对比

##### 9.1 状态数量对比

**LSTM**

- 隐藏状态 $h_t$
- 细胞状态 $C_t$

**GRU**

- 只有隐藏状态 $h_t$


##### 9.2 门控数量对比

**LSTM**

- 遗忘门
- 输入门
- 输出门

**GRU**

- 更新门
- 重置门


##### 9.3 结构复杂度对比

**LSTM**

- 更复杂
- 参数更多
- 表达能力很强
- 对长期依赖建模更精细

**GRU**

- 更简洁
- 参数更少
- 训练通常更快
- 在很多任务中效果与 LSTM 接近